# 15c · 该固定哪些位置

## 这一章在整个项目里的位置

```
第三阶段 G：从「给已有序列打分」到「自己提出新序列」
  ├─ 15  前提查验 ✅
  ├─ 15a 生成器闸门 ✅
  ├─ 15c 该固定哪些位置  ← 你在这里（本阶段的核心）
  ├─ 15b 生成 + 两个能量做选择
  ├─ 15d 平台化
  └─ 15e 讲法
```

## 背景：这一章要解决的真实问题

**场景**：有一个酶，想把它改得更结实（更耐热、更好生产），但**不能让它失去功能**。

做法是把它的形状交给生成器（ProteinMPNN），让它重写一条序列。但生成器**不知道哪些位置是干活用的**，
所以要事先告诉它"这些位置别动"。

**问题就在这里：怎么决定哪些位置别动？**

### 别人是怎么做的

Baker 实验室 2024 年（Sumida 等）在 TEV 蛋白酶（221 个位置）上做过一次很干净的实验：

| 固定哪些位置 | 固定几个 | 做出来的蛋白 |
|---|---|---|
| 按**三维距离**圈出的活性位点 | 38 | **24 条，全部没活性** |
| 再加最保守的 30% | 66 | **24 条，全部没活性** |
| 再加到最保守的 **50%** | **127** | **这才有活性** |

两个结论：

1. **按距离圈活性位点，完全不够。**
2. 补救办法是叠加"**进化保守性**"，但那个 50% 是**试出来的**——作者明说每个蛋白都得重新试。

### 我们能插进来的地方

我们有一个别人没有的东西：**每个位置的功能代价，带 kcal/mol 单位**。

所以这一章问一个文献里没人问过的问题：

> **固定同样多的位置，按什么排序挑，效果最好？**

---

## 怎么验证（不需要湿实验）

**关键想法：不用真做蛋白，用别人已经测过的突变当答案本。**

有些蛋白，每个位置的每种替换都被测过——测出"改了这里，功能掉多少"。

于是检验变成：

```
每种排序方法挑出 N 个位置固定住
→ 翻答案本：被放开的那些位置，实测损伤有多大
→ 损伤最小的方法赢
```

**我们能验的是"哪些位置该固定"这个判断准不准**，这正是 Baker 实验室栽跟头的那一环。
**验不了**"这条设计做出来真有活性"——那需要湿实验。

### 参赛的六种排序

| 方法 | 怎么排 | 谁在用 |
|---|---|---|
| **我们 功能能** | 每位点的功能代价 | 本项目 |
| **我们 折叠能** | 每位点的折叠代价 | 本项目 |
| 我们 两个相加 | 两个代价之和 | 本项目 |
| **保守性** | ESM-2 在这个位置的熵，越低越保守 | **Sumida 最后用的** |
| ESM-2 替换代价 | 语言模型认为这里的替换有多糟 | 常见基线 |
| **埋藏程度** | 埋得越深越重要 | **Sumida 第一次用的（距离的近亲）** |
| **生成器自己** | ProteinMPNN 自己的分布熵 | **「只用生成器」的消融** |

最后那一行是关键：**它回答"不加我们的东西，生成器自己行不行"。**

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
print("benchmark:", (ROOT / "data/sites/ranking_benchmark.parquet").exists(),
      "| kras:", (ROOT / "data/kras/kras_position_benchmark.parquet").exists())

---

## 第 1 步 · 大样本：146 组实验、107 个蛋白

### 在干什么
在每一组实验内部，看每种排序和**实测损伤**的相关性，再取所有实验的中位数。

### 真值是什么
每个位置上所有替换的实测读数的平均。读数 1 附近=正常，0=失活，**所以读数低 = 损伤大**。

In [ ]:
d = pd.read_parquet("data/sites/ranking_benchmark.parquet")
RANKINGS = {
    "我们 功能能": "rank_func", "我们 折叠能": "rank_fold", "我们 两个相加": "rank_both",
    "保守性 (ESM 熵)": "rank_conservation", "ESM-2 替换代价": "rank_esm_delta",
    "埋藏程度 (RSA)": "rank_burial", "生成器自己 (MPNN 熵)": "rank_mpnn",
}

rows = []
for name, col in RANKINGS.items():
    rhos = [-stats.spearmanr(g.measured, g[col]).statistic
            for _, g in d.groupby("dataset_id")
            if g[col].notna().all() and g.measured.nunique() > 2]
    rows.append({"排序方法": name, "中位 Spearman": round(float(np.median(rhos)), 3), "实验数": len(rhos)})
print(pd.DataFrame(rows).sort_values("中位 Spearman", ascending=False).to_string(index=False))

### 结果怎么看

| 排序方法 | 中位 Spearman |
|---|---|
| **我们 折叠能** | **0.588** |
| 我们 两个相加 | 0.581 |
| **生成器自己 (MPNN)** | **0.516** |
| 埋藏程度 | 0.493 |
| 我们 功能能 | 0.454 |
| ESM-2 替换代价 | 0.453 |
| **保守性** | **0.364** |

**第一个消融结果**：我们的能量（0.588）**赢过只用生成器（0.516）**。

**但这张表有个必须说破的问题**：146 组实验里 **127 组是 domainome**，测的是"细胞里还剩多少蛋白"
——那本来就是**折叠**。折叠能在这里赢，接近同义反复。

**真正要考的"功能位置挑得准不准"，得换考场。** 先按实验类型拆开看。

In [ ]:
out = []
for kind, g in d.groupby("data_kind"):
    row = {"实验类型": kind, "组数": g.dataset_id.nunique()}
    for name, col in RANKINGS.items():
        rhos = [-stats.spearmanr(x.measured, x[col]).statistic
                for _, x in g.groupby("dataset_id")
                if x[col].notna().all() and x.measured.nunique() > 2]
        row[name] = round(float(np.median(rhos)), 3) if rhos else None
    out.append(row)
print(pd.DataFrame(out).to_string(index=False))

### 拆开之后

| 实验类型 | 组数 | 测的是 | 谁赢 | 生成器 |
|---|---|---|---|---|
| **domainome** | 127 | 蛋白量（≈折叠） | 我们折叠能 0.586 | 0.521（第 2） |
| **mavedb** | 16 | 混合，含活性 | **两个相加 0.638** | **0.427（倒数第 1）** |
| proteingym | 3 | 稳定性 | 两个相加 0.764 | 0.665 |

**关键在中间那一行**：一旦实验里含**活性**，**生成器就从第 2 名掉到最后一名**。

这正是文献里那个已知毛病的量化——**它懂结构，不懂功能**。

In [ ]:
pairs = []
for _, x in d.groupby("dataset_id"):
    if x[["rank_func", "rank_conservation"]].notna().all().all() and x.measured.nunique() > 2:
        pairs.append((-stats.spearmanr(x.measured, x.rank_func).statistic,
                      -stats.spearmanr(x.measured, x.rank_conservation).statistic))
a = np.array(pairs)
w = stats.wilcoxon(a[:, 0], a[:, 1])
print(f"配对比较（同一组实验上）：n = {len(a)} 组")
print(f"  我们功能能 中位 {np.median(a[:, 0]):.3f}   保守性 中位 {np.median(a[:, 1]):.3f}")
print(f"  我们更好的组数 {int((a[:, 0] > a[:, 1]).sum())}/{len(a)}   Wilcoxon p = {w.pvalue:.2e}")

### 和 Sumida 用的那个方法正面比

**我们的功能能 0.454 vs 保守性 0.364，146 组里赢 107 组，p = 6.3e-11。**

这是这一章最硬的一个数：**Sumida 最后靠"固定最保守的 50%"才救回活性，而那个排序依据，
在 146 组实测上被我们的功能代价显著超过。**

---

## 第 2 步 · 功能轴的干净考场：KRAS

### 为什么要单独做这个
上一步的考场被"测蛋白量"的实验主导。**KRAS 是唯一一个两件事都被测成能量的考试蛋白**：
别人同时测了它的**折叠**和它跟 **6 个搭档的结合**。而且它和它唯一的近亲都锁在考试集里。

### 一个要小心的坑
KRAS 的实测是 **ΔΔG**（能量），**数值越高 = 损伤越大**；
而上一步的实测是适应度，**数值越高 = 越好**。两者方向相反。

写这一章时我第一遍就把两套方向用混了，整节符号翻了。用"埋藏程度"当参照才定准：
埋得深的位置突变后更坏，所以 `spearman(rsa, 实测)` 的符号能唯一确定每个实测量的方向。

In [ ]:
pos = pd.read_parquet("data/kras/kras_position_benchmark.parquet")
K = {"我们 功能能": -pos.ours_func, "我们 折叠能": -pos.ours_fold,
     "生成器 (MPNN 熵)": -pos.mpnn_entropy, "保守性 (ESM 熵)": -pos.esm_entropy,
     "ESM-2 替换代价": -pos.esm_delta, "埋藏程度 (RSA)": -pos.rsa}

rows = [{"排序方法": n,
         "对实测折叠": round(stats.spearmanr(pos.meas_fold, r).statistic, 3),
         "对实测结合": round(stats.spearmanr(pos.meas_bind, r).statistic, 3)} for n, r in K.items()]
t = pd.DataFrame(rows)
t["结合 − 折叠"] = (t["对实测结合"] - t["对实测折叠"]).round(3)
print(f"KRAS，{len(pos)} 个位置")
print(t.sort_values("对实测结合", ascending=False).to_string(index=False))

### 结果怎么看：这张表是整个第三阶段的立论依据

| 排序方法 | 对实测**折叠** | 对实测**结合** | 偏向 |
|---|---|---|---|
| ESM-2 替换代价 | 0.359 | **0.563** | 偏功能 |
| **我们 功能能** | 0.261 | **0.508** | 偏功能 |
| 保守性 | 0.214 | 0.498 | 偏功能 |
| **我们 折叠能** | **0.460** | 0.312 | 偏结构 |
| 埋藏程度 | 0.407 | 0.222 | 偏结构 |
| **生成器 (MPNN)** | **0.425** | **0.161** | **最偏结构** |

**生成器在折叠上排第 2（0.425），在结合上排最后（0.161）。**

**我们的两个能量各占一头**：折叠能是折叠轴第一（0.460），功能能是结合轴第二（0.508）。

这就是"生成器需要一个功能信号"这句话的量化版本。

In [ ]:
rows = []
for n, r in K.items():
    n1, n2 = int(pos.is_functional.sum()), int((~pos.is_functional).sum())
    u = stats.mannwhitneyu(r[pos.is_functional], r[~pos.is_functional], alternative="greater")
    rows.append({"排序方法": n, "AUROC": round(u.statistic / (n1 * n2), 3), "p": f"{u.pvalue:.1e}"})
print(f"真正的功能位置（结合坏了但折叠没坏）：{int(pos.is_functional.sum())} / {len(pos)}\n")
print(pd.DataFrame(rows).sort_values("AUROC", ascending=False).to_string(index=False))

### 最尖锐的那个问题

把"**结合损伤超出折叠能解释的部分**"最严重的 20%（34 个位置）定义为真正的功能位置——
这些就是**结构上看不出来、但动了就不干活**的位置，也正是 Baker 实验室栽跟头的那一类。

| 排序方法 | AUROC |
|---|---|
| ESM-2 替换代价 | 0.785 |
| **我们 功能能** | **0.781** |
| 保守性 | 0.780 |
| 我们 折叠能 | 0.614 |
| **生成器 (MPNN)** | **0.532**（p = 0.28，**不显著**） |
| 埋藏程度 | 0.530（不显著） |

**生成器在这件事上等于瞎猜。** 而"按距离/埋藏程度圈活性位点"也等于瞎猜（0.530）——
**这就是 Sumida 那 24 条设计全军覆没的原因，在数据上看得见。**

### 一条必须直说的：在 KRAS 上我们没赢保守性

0.781 对 0.780 对 0.785，**三者并列**。

n = 166、单个蛋白，平手不奇怪。**但不能把平手说成赢。**
"我们比保守性准"这个结论的依据是上一步的 146 组配对检验（p = 6.3e-11），**不是 KRAS 这一步**。

---

## 这一章的结论

### 站得住的

✅ **只用生成器，找不到功能位置。** KRAS 上 AUROC 0.532（p = 0.28，等于瞎猜）；
一旦实验含活性，它从第 2 名掉到最后一名（mavedb 组 0.427）。

✅ **按距离/埋藏程度圈活性位点也不行**（AUROC 0.530，不显著）——
这正是 Sumida 第一轮 24 条设计全军覆没的原因。

✅ **我们的功能能显著优于 Sumida 最后采用的保守性排序**：146 组配对，107 组更好，p = 6.3e-11。

✅ **两个能量各管一头**：折叠能是折叠轴第一，功能能是结合轴第二。

### 站不住的（不许含糊）

❌ **在 KRAS 的功能轴上，我们和保守性、ESM-2 是平手**（0.781 / 0.780 / 0.785）。

❌ **验不了"这条设计做出来真有活性"**——需要湿实验。

❌ **我们的模型输入里本来就含 ESM-2 的信号**，所以这不是"我们 vs 保守性"的纯粹对决，
而是"保守性 vs 保守性 + 结构 + 有监督训练"。这一点在引用时必须带上。

## 泄漏自查

| 检查项 | 结果 |
|---|---|
| 基准里的 107 个蛋白全在 test split 吗 | ✅ 107/107，train 里 0 个 |
| KRAS 在 test split 吗 | ✅ |
| 实测标签有没有参与拟合 | ✅ 没有，本章不训练任何模型 |
| 各方法的预训练暴露面 | ESM-2 见过考试蛋白序列、ProteinMPNN 见过其结构——**但基线和我们都一样**，不构成不公平 |
| 我们的模型 | 只在 train split 的蛋白上训练过 |

## 你现在要做什么

**什么都不用做。** 这一章是分析，没有要你操作的东西。

## 记账

数字由本 notebook 从 `data/sites/ranking_benchmark.parquet` 和
`data/kras/kras_position_benchmark.parquet` 重算，这两份由本章的分析脚本生成。
**MLflow 还没记**——这一章不训练模型。